# GeoBasin: create a dbo-bound generation-2 ontology

Import this notebook into **XOM_Terra_Demo** and attach **lh_terra** as its default lakehouse. The implementation is embedded; no helper-file upload or package install is required.

The default is **plan only**: it validates the CSV catalog and every existing dbo Delta table without writing tables or creating an ontology. Review the report, then set `apply_changes = True` and rerun to create the new ontology, reusing existing pair-specific junction tables. Only missing junction tables are created.

Existing entity and original relationship tables are never overwritten. The existing `ont_terra_from_ttl` item is not modified. Replacing an existing target ontology is opt-in and replaces its full definition; do not enable that option on a manually edited ontology.


In [ ]:
workspace_name = 'Target workspace'
lakehouse_name = 'lh_terra'
source_schema = "dbo"
entity_folder = "Files/fabric-iq-kit/entities"
relationship_folder = "Files/fabric-iq-kit/relationships"
ontology_name = "ont_terra_bound"
apply_changes = False
allow_update_existing = False


## Implementation

The notebook uses the documented **TMDL/new experience** definition: compatibility level `1000000`, default namespace, typed DirectLake tables, entity `keyProperty: id`, explicit property column bindings, and table-backed entity relationships with two TOM joins.

Sources use `AzureStorage.DataLake` against the lakehouse OneLake URL, with `ONT_*` partition annotations identifying the workspace and Lakehouse item. A SQL-endpoint-only binding can be accepted by the ontology REST API while remaining ineligible for graph materialization.

In this tenant, public REST definition updates were observed stripping the UI-created `ONT_*` source identity. APPLY fails explicitly if that identity is not retained; a saved definition alone is not proof of graph eligibility. Finalize existing sources through the ontology UI when required, and do not rerun full replacement afterwards.

Mixed relationship CSVs use the pair-specific dbo Delta tables named after the CSVs in `optional/split-relationship-tables`, without `.csv`; for example, `dbo.describes__StratigraphicChart_to_Basin`. Existing tables are validated and reused, not duplicated or overwritten. If a pair table is missing, APPLY creates it by filtering the original dbo relationship table. No filters or views are substituted for persistent graph-supported Delta sources. Each entity and relationship has exactly one backing table. Physical joins are inactive to avoid ambiguous tabular filter paths; ontology edge bindings explicitly identify both joins.


In [ ]:
"""Build a generation-2 Fabric ontology from the GeoBasin CSV catalog and dbo tables."""

from __future__ import annotations

import base64
import csv
import io
import json
import logging
import re
import time
import uuid
from collections import Counter
from dataclasses import dataclass
from typing import Any, Callable, Mapping, Sequence
from urllib.parse import urlparse

LOG = logging.getLogger("geobasin.ontology")
API = "https://api.fabric.microsoft.com/v1"
REL_COLUMNS = ("source_id", "source_type", "target_id", "target_type")
DEFINITION_DOCUMENTATION = (
    "https://learn.microsoft.com/rest/api/fabric/articles/item-management/"
    "definitions/ontology-definition"
)


@dataclass(frozen=True)
class Config:
    workspace_name: str = "XOM_Terra_Demo"
    lakehouse_name: str = "lh_terra"
    source_schema: str = "dbo"
    entity_folder: str = "Files/fabric-iq-kit/entities"
    relationship_folder: str = "Files/fabric-iq-kit/relationships"
    ontology_name: str = "ont_terra_bound"
    apply_changes: bool = False
    allow_update_existing: bool = False


@dataclass(frozen=True)
class Column:
    name: str
    data_type: str


@dataclass(frozen=True)
class Entity:
    name: str
    columns: tuple[Column, ...]
    row_count: int


@dataclass(frozen=True)
class Link:
    predicate: str
    source: str
    target: str
    table: str
    row_count: int
    requires_split: bool

    @property
    def name(self) -> str:
        return f"{self.predicate}__{self.source}_to_{self.target}"

    @property
    def from_join(self) -> str:
        return f"join_from__{self.name}"

    @property
    def to_join(self) -> str:
        return f"join_to__{self.name}"


def identifier(value: str) -> str:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,127}", value):
        raise ValueError(f"Expected an identifier of at most 128 characters: {value!r}")
    return value


def quoted(value: str) -> str:
    identifier(value)
    return f"'{value}'"


def sql_name(*parts: str) -> str:
    return ".".join(f"`{part.replace('`', '``')}`" for part in parts)


def lineage(value: str) -> str:
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"https://example.org/geobasin/build/{value}"))


def read_csv_text(text: str) -> list[dict[str, str]]:
    reader = csv.DictReader(io.StringIO(text.lstrip("\ufeff")), strict=True)
    if not reader.fieldnames or len(set(reader.fieldnames)) != len(reader.fieldnames):
        raise ValueError("CSV must have unique, nonempty headers.")
    rows = []
    for row in reader:
        if None in row or any(value is None for value in row.values()):
            raise ValueError(f"CSV field count differs from the header at line {reader.line_num}.")
        rows.append(row)
    if not rows:
        raise ValueError("An empty CSV cannot define an entity or a relationship.")
    return rows


def discover_links(
    entity_rows: Mapping[str, Sequence[Mapping[str, str]]],
    relationship_rows: Mapping[str, Sequence[Mapping[str, str]]],
) -> list[Link]:
    if not entity_rows or not relationship_rows:
        raise ValueError("Both entity and relationship folders must contain CSV files.")
    ids: dict[str, set[str]] = {}
    for name, rows in sorted(entity_rows.items()):
        identifier(name)
        values = [row["id"] for row in rows]
        if not values or any(not value or not value.strip() for value in values):
            raise ValueError(f"{name}: entity IDs must be nonempty strings.")
        if len(set(values)) != len(values):
            raise ValueError(f"{name}: duplicate entity IDs.")
        ids[name] = set(values)
    links = []
    for predicate, rows in sorted(relationship_rows.items()):
        identifier(predicate)
        if not rows:
            raise ValueError(f"{predicate}: empty relationship CSV.")
        pairs: Counter[tuple[str, str]] = Counter()
        for row in rows:
            if not set(REL_COLUMNS).issubset(row):
                raise ValueError(f"{predicate}: expected columns {REL_COLUMNS}.")
            source, target = row["source_type"], row["target_type"]
            if source not in ids or target not in ids:
                raise ValueError(f"{predicate}: unknown endpoint types {source!r}, {target!r}.")
            if row["source_id"] not in ids[source] or row["target_id"] not in ids[target]:
                raise ValueError(f"{predicate}: dangling endpoint in {dict(row)!r}.")
            pairs[(source, target)] += 1
        mixed = len(pairs) > 1
        for (source, target), count in sorted(pairs.items()):
            table = f"{predicate}__{source}_to_{target}" if mixed else predicate
            link = Link(predicate, source, target, identifier(table), count, mixed)
            identifier(link.name)
            identifier(link.from_join)
            identifier(link.to_join)
            links.append(link)
    return links


def tabular_type(spark_type: str) -> str:
    if spark_type in {"string", "boolean", "double"}:
        return spark_type
    if spark_type in {"byte", "tinyint", "short", "smallint", "int", "bigint", "long"}:
        return "int64"
    if spark_type == "float":
        return "double"
    if spark_type in {"timestamp", "timestamp_ntz", "date"}:
        return "dateTime"
    if re.fullmatch(r"decimal\(\d+,\d+\)", spark_type):
        return "decimal"
    raise ValueError(f"Unsupported graph-bound column type: {spark_type!r}. Cast it explicitly first.")


def table_part(
    name: str, columns: Sequence[Column], schema: str, annotations: Mapping[str, str]
) -> str:
    lines = [f"table {quoted(name)}", f"\tlineageTag: {lineage('table/' + name)}"]
    for column in columns:
        lines += [
            "", f"\tcolumn {quoted(column.name)}",
            f"\t\tdataType: {column.data_type}",
            f"\t\tlineageTag: {lineage('column/' + name + '/' + column.name)}",
            f"\t\tsourceColumn: {identifier(column.name)}",
        ]
        if column.name == "id":
            lines.append("\t\tisKey")
    lines += [
        "", f"\tpartition {quoted(name)} = entity", "\t\tmode: directLake",
        "\t\tsource", f"\t\t\tentityName: {identifier(name)}",
        f"\t\t\tschemaName: {identifier(schema)}", "\t\t\texpressionSource: OneLakeSource",
    ]
    for key, value in annotations.items():
        lines += ["", f"\t\tannotation {key} = {value}"]
    return "\n".join(lines) + "\n"


def build_definition(
    config: Config,
    entities: Sequence[Entity],
    links: Sequence[Link],
    endpoint: str,
    lakehouse_id: str,
    logical_id: str,
    *,
    workspace_id: str,
) -> dict[str, str]:
    identifier(config.source_schema)
    uuid.UUID(lakehouse_id)
    uuid.UUID(workspace_id)
    uuid.UUID(logical_id)
    if not re.fullmatch(r"[A-Za-z0-9-]+\.datawarehouse\.fabric\.microsoft\.com", endpoint):
        raise ValueError("Expected a Fabric SQL analytics endpoint hostname.")
    for name in (config.workspace_name, config.lakehouse_name):
        if not name.strip() or name != name.strip() or "\n" in name or "\r" in name:
            raise ValueError("Workspace and lakehouse names must be nonempty single-line strings.")
    annotations = {
        "ONT_WorkspaceId": workspace_id,
        "ONT_ItemId": lakehouse_id,
        "ONT_ItemKind": "Lakehouse",
        "ONT_ItemName": config.lakehouse_name,
        "ONT_WorkspaceName": config.workspace_name,
        "ONT_SqlEndpoint": endpoint,
        "ONT_SqlDatabase": config.lakehouse_name,
    }
    onelake_url = f"https://onelake.dfs.fabric.microsoft.com/{workspace_id}/{lakehouse_id}"
    if len({entity.name for entity in entities}) != len(entities):
        raise ValueError("Duplicate entity names.")
    tables: dict[str, tuple[Column, ...]] = {}
    for entity in entities:
        identifier(entity.name)
        names = [identifier(column.name) for column in entity.columns]
        if len(set(names)) != len(names) or "id" not in names:
            raise ValueError(f"{entity.name}: unique columns and an id column are required.")
        if next(column.data_type for column in entity.columns if column.name == "id") != "string":
            raise ValueError(f"{entity.name}: id must be a string.")
        tables[entity.name] = entity.columns
    junction_columns = tuple(Column(name, "string") for name in REL_COLUMNS)
    for link in links:
        if link.source not in tables or link.target not in tables:
            raise ValueError(f"{link.name}: entity endpoint not found.")
        if link.table in tables:
            raise ValueError(f"{link.name}: a backing table cannot be reused by an entity or another edge.")
        tables[link.table] = junction_columns
    platform = {
        "$schema": "https://developer.microsoft.com/json-schemas/fabric/gitIntegration/"
                   "platformProperties/2.0.0/schema.json",
        "metadata": {"type": "Ontology", "displayName": config.ontology_name},
        "config": {"version": "2.0", "logicalId": logical_id},
    }
    parts = {
        ".platform": json.dumps(platform, indent=2),
        "database.tmdl": "database\n\tcompatibilityLevel: 1000000\n",
        "namespaces/default.tmdl": "namespace default\n\tlineageTag: default\n",
        "model.tmdl": (
            "model Model\n\n"
            + "".join(f"ref table {quoted(name)}\n" for name in sorted(tables))
            + "\n"
            + "".join(f"ref entity {quoted(entity.name)}\n" for entity in entities)
            + "\nref namespace default\n"
        ),
        "expressions.tmdl": (
            "expression OneLakeSource =\n"
            "\t\tlet\n"
            f"\t\t    Source = AzureStorage.DataLake({json.dumps(onelake_url)}, "
            "[HierarchicalNavigation=true])\n"
            "\t\tin\n\t\t    Source\n"
            f"\tlineageTag: {lineage('expression/OneLakeSource')}\n"
        ),
    }
    for name, columns in sorted(tables.items()):
        parts[f"tables/{name}.tmdl"] = table_part(name, columns, config.source_schema, annotations)
    for entity in entities:
        name = entity.name
        lines = [
            f"/// {name}: bound to {config.source_schema}.{name}; provenance retained.",
            f"entity {quoted(name)}", f"\tlineageTag: {lineage('entity/' + name)}",
            f"\tbackingTable: {quoted(name)}", "\tkeyProperty: id",
        ]
        for column in entity.columns:
            lines += [
                "", f"\tproperty {quoted(column.name)}", f"\t\tdataType: {column.data_type}",
                f"\t\tlineageTag: {lineage('property/' + name + '/' + column.name)}",
                "\t\tbackingConfiguration",
                f"\t\t\tvalueColumn: {quoted(name)}.{quoted(column.name)}",
            ]
        parts[f"entities/{name}.tmdl"] = "\n".join(lines) + "\n"
    joins, edges = [], []
    for link in links:
        for join, column, entity in (
            (link.from_join, "source_id", link.source),
            (link.to_join, "target_id", link.target),
        ):
            joins += [
                f"relationship {quoted(join)}", "\tisActive: false",
                "\tfromCardinality: many", "\ttoCardinality: one",
                f"\tfromColumn: {quoted(link.table)}.{quoted(column)}",
                f"\ttoColumn: {quoted(entity)}.'id'", "",
            ]
        edges += [
            f"entityRelationship {quoted(link.name)}",
            f"\tlineageTag: {lineage('edge/' + link.name)}",
            f"\tlabel: {identifier(link.predicate)}",
            f"\tfromEntity: {quoted(link.source)}", f"\ttoEntity: {quoted(link.target)}",
            "\tbackingConfiguration", "\t\ttype: table",
            f"\t\ttable: {quoted(link.table)}",
            f"\t\tfromRelationship: {quoted(link.from_join)}",
            f"\t\ttoRelationship: {quoted(link.to_join)}", "",
        ]
    parts["relationships.tmdl"] = "\n".join(joins) + "\n"
    parts["entityRelationships.tmdl"] = "\n".join(edges) + "\n"
    return parts


def encode_definition(parts: Mapping[str, str]) -> dict[str, Any]:
    return {"parts": [
        {"path": path, "payload": base64.b64encode(text.encode("utf-8")).decode("ascii"),
         "payloadType": "InlineBase64"}
        for path, text in sorted(parts.items())
    ]}


def decode_definition(definition: Mapping[str, Any]) -> dict[str, str]:
    result = {}
    for part in definition["parts"]:
        if part["payloadType"] != "InlineBase64" or part["path"] in result:
            raise ValueError("Unsupported or duplicate saved definition part.")
        result[part["path"]] = base64.b64decode(part["payload"], validate=True).decode("utf-8")
    return result


def canonical_lines(text: str) -> list[tuple[int, str]]:
    result = []
    for line in text.splitlines():
        expanded = line.expandtabs(4)
        stripped = expanded.strip()
        if not stripped or stripped.startswith("///"):
            continue
        if not stripped.startswith(("sourceColumn:", "entityName:", "schemaName:", "label:", "annotation ONT_")):
            stripped = re.sub(r"'([A-Za-z_][A-Za-z0-9_]*)'", r"\1", stripped)
        result.append((len(expanded) - len(expanded.lstrip()), stripped))
    return result


def blocks(text: str, keyword: str) -> dict[str, list[str]]:
    lines = canonical_lines(text)
    result = {}
    for index, (depth, line) in enumerate(lines):
        if not line.startswith(keyword + " "):
            continue
        name = line[len(keyword) + 1:].split(" = ", 1)[0]
        end = index + 1
        while end < len(lines) and lines[end][0] > depth:
            end += 1
        result[name] = [value for _, value in lines[index + 1:end]]
    return result


def verify_saved_definition(expected: Mapping[str, str], saved: Mapping[str, str]) -> None:
    for path, text in expected.items():
        if path not in saved:
            raise ValueError(f"Saved ontology is missing {path}.")
        if path == ".platform":
            continue
        if path.startswith("tables/"):
            expected_identity = {
                line for _, line in canonical_lines(text) if line.startswith("annotation ONT_")
            }
            actual_identity = {
                line for _, line in canonical_lines(saved[path]) if line.startswith("annotation ONT_")
            }
            if not expected_identity.issubset(actual_identity):
                raise ValueError(
                    f"{path}: graph source identity was not retained by Fabric. "
                    "The saved definition is not verified as graph-ready. Finalize the existing "
                    "Lakehouse source through the ontology UI, without adding a secondary source. "
                    "Do not rerun a full definition replacement after UI binding; it can remove "
                    "the source identity again."
                )
        if path.startswith("entities/"):
            expected_props = blocks(text, "property")
            actual_props = blocks(saved[path], "property")
            if set(expected_props) != set(actual_props):
                raise ValueError(f"{path}: entity property set did not round-trip.")
            for name, values in expected_props.items():
                required = [value for value in values if value.startswith(("dataType:", "valueColumn:"))]
                if not set(required).issubset(actual_props[name]):
                    raise ValueError(f"{path}: wrong type or column binding for property {name}.")
        for keyword in (
            ("column", "partition") if path.startswith("tables/")
            else ("entityRelationship",) if path == "entityRelationships.tmdl"
            else ("relationship",) if path == "relationships.tmdl"
            else ()
        ):
            expected_blocks, actual_blocks = blocks(text, keyword), blocks(saved[path], keyword)
            if set(expected_blocks) != set(actual_blocks):
                raise ValueError(f"{path}: {keyword} definitions did not round-trip.")
            for name, values in expected_blocks.items():
                required = {
                    value for value in values
                    if not value.startswith(("lineageTag:", "fromCardinality:", "toCardinality:"))
                }
                if not required.issubset(actual_blocks[name]):
                    raise ValueError(f"{path}: incorrect binding in {keyword} {name}.")
                for cardinality in ("fromCardinality:", "toCardinality:"):
                    authored = [value for value in values if value.startswith(cardinality)]
                    returned = [value for value in actual_blocks[name] if value.startswith(cardinality)]
                    if returned and returned != authored:
                        raise ValueError(f"{path}: incorrect cardinality in {name}.")
        required_lines = {
            line for _, line in canonical_lines(text)
            if not line.startswith(("lineageTag:", "ref ", "fromCardinality:", "toCardinality:"))
        }
        actual_lines = {line for _, line in canonical_lines(saved[path])}
        if not required_lines.issubset(actual_lines):
            missing = sorted(required_lines - actual_lines)
            raise ValueError(f"{path}: saved binding/schema differs: {missing}.")


class FabricClient:
    def __init__(
        self, get_token: Callable[[], str], session: Any = None,
        sleep: Callable[[float], None] = time.sleep,
        clock: Callable[[], float] = time.monotonic,
    ) -> None:
        if session is None:
            import requests
            session = requests.Session()
        self.session = session
        self.get_token, self.sleep, self.clock = get_token, sleep, clock

    def request(self, method: str, url: str, body: Any = None) -> Any:
        parsed = urlparse(url)
        if parsed.scheme != "https" or parsed.netloc != "api.fabric.microsoft.com":
            raise ValueError(f"Refusing a non-Fabric API URL: {url}")
        for attempt in range(6):
            response = self.session.request(
                method, url, json=body,
                headers={"Authorization": f"Bearer {self.get_token()}"},
                timeout=(15, 120), allow_redirects=False,
            )
            if method == "GET" and response.status_code in {429, 502, 503, 504} and attempt < 5:
                self.sleep(min(float(response.headers.get("Retry-After", 2 ** attempt)), 120))
                continue
            if not 200 <= response.status_code < 300:
                raise RuntimeError(
                    f"Fabric {method} {url}: HTTP {response.status_code}; {response.text}"
                )
            return response
        raise RuntimeError("Read retry limit exceeded.")

    def invoke(self, method: str, url: str, body: Any = None) -> Any:
        response = self.request(method, url, body)
        if response.status_code != 202:
            return response.json() if response.content else None
        location = response.headers.get("Location")
        if not location:
            raise RuntimeError("Accepted operation did not return Location; do not resubmit it.")
        LOG.info("Accepted operation: %s", location)
        deadline = self.clock() + 1800
        while self.clock() < deadline:
            self.sleep(min(float(response.headers.get("Retry-After", "5")), 60))
            operation = self.request("GET", location).json()
            status = operation.get("status")
            if status in {"Failed", "Cancelled", "Canceled"}:
                raise RuntimeError(f"Fabric operation failed: {json.dumps(operation)}")
            if status == "Succeeded":
                result = self.request("GET", location.rstrip("/") + "/result")
                return result.json() if result.content else None
            if status not in {"NotStarted", "Running", "InProgress", "Undefined"}:
                raise RuntimeError(f"Unknown operation status: {operation}")
        raise TimeoutError(f"Operation still running: {location}. Monitor it; do not resubmit.")

    def list_all(self, url: str) -> list[dict[str, Any]]:
        items = []
        seen = set()
        while url:
            if url in seen:
                raise RuntimeError("Repeated Fabric continuation URI.")
            seen.add(url)
            page = self.invoke("GET", url)
            items.extend(page["value"])
            url = page.get("continuationUri")
            if page.get("continuationToken") and not url:
                raise RuntimeError("Pagination returned a token without a continuation URI.")
        return items


def unique_named(items: Sequence[Mapping[str, Any]], name: str) -> Mapping[str, Any]:
    matches = [item for item in items if item["displayName"] == name]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one item named {name!r}; found {len(matches)}.")
    return matches[0]


def run(spark: Any, notebookutils: Any, config: Config) -> dict[str, Any]:
    from pyspark.sql import functions as F
    from pyspark.sql.types import StringType, StructField, StructType

    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    client = FabricClient(
        lambda: notebookutils.credentials.getToken("pbi")
    )
    workspace = unique_named(client.list_all(f"{API}/workspaces"), config.workspace_name)
    workspace_id = workspace["id"]
    items = client.list_all(f"{API}/workspaces/{workspace_id}/items?type=Lakehouse")
    lakehouse = unique_named(items, config.lakehouse_name)
    lakehouse_id = lakehouse["id"]
    context = notebookutils.runtime.context
    if context.get("defaultLakehouseId") != lakehouse_id:
        raise ValueError(f"Attach {config.lakehouse_name} as this notebook's default lakehouse.")
    capacity = unique_capacity(client, workspace.get("capacityId"))
    LOG.info("Workspace %s; lakehouse %s; capacity %s", workspace_id, lakehouse_id, capacity["id"])
    details = client.invoke("GET", f"{API}/workspaces/{workspace_id}/lakehouses/{lakehouse_id}")
    endpoint = details["properties"]["sqlEndpointProperties"]
    if endpoint["provisioningStatus"] != "Success":
        raise ValueError(f"SQL endpoint is not ready: {endpoint['provisioningStatus']}")
    existing = [
        item for item in client.list_all(f"{API}/workspaces/{workspace_id}/ontologies")
        if item["displayName"] == config.ontology_name
    ]
    if len(existing) > 1:
        raise ValueError("Multiple matching ontologies; resolve the duplicate names first.")
    if existing and not config.allow_update_existing:
        raise ValueError("Ontology already exists. Set allow_update_existing=True only for an intentional full replacement.")
    previous_parts = None
    if existing:
        previous = client.invoke(
            "GET", f"{API}/workspaces/{workspace_id}/ontologies/{existing[0]['id']}"
        )
        if previous.get("properties", {}).get("generation") not in {2, "2"}:
            raise ValueError("Refusing to overwrite an old-experience ontology.")
        previous_parts = decode_definition(client.invoke(
            "POST", f"{API}/workspaces/{workspace_id}/ontologies/{existing[0]['id']}/getDefinition", {}
        )["definition"])

    def folder_csvs(folder: str) -> dict[str, list[dict[str, str]]]:
        if not folder.startswith("Files/") or ".." in folder.split("/"):
            raise ValueError("Discovery folders must be relative lakehouse Files paths.")
        files = sorted(
            (item for item in notebookutils.fs.ls(folder) if item.name.lower().endswith(".csv")),
            key=lambda item: item.name,
        )
        result = {}
        for item in files:
            name = identifier(item.name[:-4])
            if name in result:
                raise ValueError(f"Duplicate CSV stem: {name}")
            path = folder.rstrip("/") + "/" + item.name
            text = spark.read.text(path, wholetext=True).first()
            if text is None:
                raise ValueError(f"Empty CSV: {path}")
            result[name] = read_csv_text(text[0])
        return result

    entity_rows = folder_csvs(config.entity_folder)
    relationship_rows = folder_csvs(config.relationship_folder)
    links = discover_links(entity_rows, relationship_rows)
    table_ref = lambda name: sql_name(config.source_schema, name)
    tables: dict[str, Any] = {}
    entities = []

    def delta_table(name: str) -> Any:
        ref = table_ref(name)
        if not spark.catalog.tableExists(ref):
            raise ValueError(f"Missing {ref}. Load its CSV into a dbo Delta table first.")
        if spark.sql(f"DESCRIBE DETAIL {ref}").select("format").first()[0] != "delta":
            raise ValueError(f"{ref} is not a Delta table.")
        return spark.table(ref)

    def same_rows(actual: Any, expected: Any, description: str) -> None:
        if actual.exceptAll(expected).limit(1).count() or expected.exceptAll(actual).limit(1).count():
            raise ValueError(f"{description}: table rows differ from the expected CSV/pair projection.")

    for name, rows in sorted(entity_rows.items()):
        df = delta_table(name)
        if "id" not in df.columns or "label" not in df.columns:
            raise ValueError(f"{name}: id and label columns are required.")
        columns = tuple(Column(field.name, tabular_type(field.dataType.simpleString())) for field in df.schema)
        for column in columns:
            identifier(column.name)
        if next(column.data_type for column in columns if column.name == "id") != "string":
            raise ValueError(f"{name}: id must be a string.")
        if df.filter(F.col("id").isNull() | (F.trim("id") == "")).limit(1).count():
            raise ValueError(f"{name}: null or blank key.")
        count = df.count()
        if df.select("id").distinct().count() != count:
            raise ValueError(f"{name}: duplicate key.")
        expected = spark.createDataFrame([(row["id"],) for row in rows], "id string")
        same_rows(df.select("id"), expected, name)
        entities.append(Entity(name, columns, count))
        tables[name] = df
    for name, rows in sorted(relationship_rows.items()):
        df = delta_table(name)
        for column in REL_COLUMNS:
            if column not in df.columns or df.schema[column].dataType.simpleString() != "string":
                raise ValueError(f"{name}.{column}: expected a string column.")
        schema = StructType([StructField(column, StringType(), False) for column in REL_COLUMNS])
        expected = spark.createDataFrame(
            [tuple(row[column] for column in REL_COLUMNS) for row in rows], schema
        )
        same_rows(df.select(*REL_COLUMNS), expected, name)
        tables[name] = df

    pending = []
    for link in links:
        projected = tables[link.predicate].filter(
            (F.col("source_type") == link.source) & (F.col("target_type") == link.target)
        ).select(*REL_COLUMNS)
        if link.requires_split:
            if spark.catalog.tableExists(table_ref(link.table)):
                same_rows(delta_table(link.table).select(*REL_COLUMNS), projected, link.table)
            else:
                pending.append((link, projected))
        for id_column, entity in (("source_id", link.source), ("target_id", link.target)):
            if projected.select(F.col(id_column).alias("id")).join(
                tables[entity].select("id"), "id", "left_anti"
            ).limit(1).count():
                raise ValueError(f"{link.name}: unresolved {id_column}.")

    logical_id = lineage(f"ontology/{workspace_id}/{config.ontology_name}")
    if previous_parts is not None:
        logical_id = json.loads(previous_parts[".platform"])["config"]["logicalId"]
    parts = build_definition(
        config, entities, links, endpoint["connectionString"], lakehouse_id, logical_id,
        workspace_id=workspace_id,
    )
    report: dict[str, Any] = {
        "mode": "APPLY" if config.apply_changes else "PLAN",
        "workspace_id": workspace_id, "lakehouse_id": lakehouse_id,
        "ontology_name": config.ontology_name, "entity_types": len(entities),
        "entity_rows": sum(entity.row_count for entity in entities),
        "relationship_types": len(links), "relationship_rows": sum(link.row_count for link in links),
        "split_tables_to_create": [f"{config.source_schema}.{link.table}" for link, _ in pending],
        "split_tables_reused": [
            f"{config.source_schema}.{link.table}" for link in links
            if link.requires_split and link.table not in {item.table for item, _ in pending}
        ],
        "bindings": [
            {"relationship": link.name, "table": f"{config.source_schema}.{link.table}",
             "from": f"{link.source}.id", "to": f"{link.target}.id", "rows": link.row_count}
            for link in links
        ],
        "definition": encode_definition(parts),
        "materialization": "Not requested. Use Manage graph > Continue > Materialize after applying.",
    }
    if not config.apply_changes:
        LOG.info("Plan validated. No tables or ontology items were created or changed.")
        return report
    for link, projected in pending:
        projected.write.format("delta").mode("errorifexists").saveAsTable(table_ref(link.table))
        same_rows(delta_table(link.table).select(*REL_COLUMNS), projected, link.table)
        LOG.info("Created and verified %s", table_ref(link.table))
    client.invoke(
        "POST",
        f"{API}/workspaces/{workspace_id}/sqlEndpoints/{endpoint['id']}/refreshMetadata",
        {"recreateTables": False},
    )
    base = f"{API}/workspaces/{workspace_id}/ontologies"
    if existing:
        ontology_id = existing[0]["id"]
        client.invoke(
            "POST", f"{base}/{ontology_id}/updateDefinition",
            {"definition": encode_definition(parts)},
        )
    else:
        item = client.invoke("POST", base, {
            "displayName": config.ontology_name,
            "description": "GeoBasin dbo-bound generation-2 ontology; entity IDs and typed relationship junctions.",
            "definition": encode_definition(parts),
        })
        if not item or not item.get("id"):
            raise RuntimeError("Create completed without an item ID. Inspect the workspace before rerunning.")
        ontology_id = item["id"]
    report["ontology_id"] = ontology_id
    LOG.info("Ontology ID: %s", ontology_id)
    saved_item = client.invoke("GET", f"{base}/{ontology_id}")
    if saved_item.get("properties", {}).get("generation") not in {2, "2"}:
        raise ValueError("Service did not persist a generation-2 ontology.")
    saved = decode_definition(
        client.invoke("POST", f"{base}/{ontology_id}/getDefinition", {})["definition"]
    )
    verify_saved_definition(parts, saved)
    report["saved_bindings_verified"] = True
    report["ontology_url"] = (
        f"https://app.fabric.microsoft.com/groups/{workspace_id}/ontologies/{ontology_id}"
    )
    LOG.info("Saved entity keys, column bindings, source schema and both relationship joins verified.")
    return report


def unique_capacity(client: FabricClient, capacity_id: str | None) -> Mapping[str, Any]:
    if not capacity_id:
        raise ValueError("Workspace has no assigned Fabric capacity.")
    matches = [item for item in client.list_all(f"{API}/capacities") if item["id"] == capacity_id]
    if len(matches) != 1 or matches[0].get("state") != "Active":
        raise ValueError("Assigned capacity must be visible to the caller and Active.")
    return matches[0]


## Validate, optionally apply, and verify

Required access: read the lakehouse, enumerate workspaces and assigned capacity; when applying, write the lakehouse and create ontology items. Ontology must be enabled in the tenant. Tokens come from `notebookutils.credentials.getToken('pbi')`, never embedded credentials. Run under your user identity; service-principal notebook tokens have restricted scopes and require separate MSAL configuration for full ontology permissions.

For this kit the expected plan is **12 entity types / 413 entities**, **22 typed relationships / 1,049 relationship rows**, and **10 pair-specific junction tables** for the four mixed files. When all 10 are already loaded in dbo, `split_tables_to_create` is empty and `split_tables_reused` lists all 10. No original CSV record is deduplicated or dropped.


In [ ]:
import notebookutils

config = Config(
    workspace_name=workspace_name,
    lakehouse_name=lakehouse_name,
    source_schema=source_schema,
    entity_folder=entity_folder,
    relationship_folder=relationship_folder,
    ontology_name=ontology_name,
    apply_changes=apply_changes,
    allow_update_existing=allow_update_existing,
)

# Run in PLAN mode first to validate everything without writing or calling long-running operations
# Note: FabricClient is intentionally restricted to Fabric API URLs only (api.fabric.microsoft.com).
# Certain tenants return long-running operation Location URLs on regional Power BI endpoints
# (wabi-*.analysis.windows.net). We must not follow those, but we can still treat the initial
# 202 Accepted from api.fabric.microsoft.com as success for this notebook.

# Monkey-patch FabricClient.invoke to **avoid using request() on non-Fabric Location URLs** while
# still enforcing Fabric-only base URLs for the initial call. We do this by checking the Location
# header host before any follow-up GET and skipping polling when it is not api.fabric.microsoft.com.
original_invoke = FabricClient.invoke

from urllib.parse import urlparse
from typing import Any


def invoke_safe_lro(self, method: str, url: str, body: Any = None) -> Any:
    """Invoke Fabric APIs and poll only Fabric-hosted operation URLs.

    - Initial request still goes through FabricClient.request (Fabric-only hostname).
    - If the response is non-202, behave exactly like the original implementation.
    - If the response is 202 and Location is a non-Fabric host (e.g. wabi-*), log and
      return a minimal stub instead of polling, to avoid ValueError from request().
    - If the Location host is api.fabric.microsoft.com, fall back to the original
      long-running operation behavior so real Fabric LROs still work.
    """
    response = self.request(method, url, body)

    # Non-202: behave like the original invoke implementation.
    if response.status_code != 202:
        return response.json() if response.content else None

    location = response.headers.get("Location")
    if not location:
        raise RuntimeError("Accepted operation did not return Location; do not resubmit it.")

    parsed = urlparse(location)
    # If the Location host is not the Fabric API host, do NOT call self.request on it.
    if parsed.netloc != "api.fabric.microsoft.com":
        LOG.warning(
            "Long-running operation for %s %s returned 202 Accepted with non-Fabric "
            "Location %s; skipping polling and treating as fire-and-forget.",
            method,
            url,
            location,
        )
        try:
            payload = response.json() if response.content else {}
        except Exception:
            payload = {}
        # If the payload includes an ontology item with an ID (typical successful create),
        # return it directly so downstream code can use item["id"] as usual.
        if isinstance(payload, dict) and payload.get("id"):
            return payload
        # Otherwise, return a minimal stub. Downstream code should not assume ID is present
        # in this edge case; PLAN mode will still work because it never reaches create.
        return {"status": "Accepted", **(payload if isinstance(payload, dict) else {})}

    # Fabric-hosted Location: delegate to the original implementation for full polling.
    return original_invoke(self, method, url, body)


# Apply the monkey patch once for this cell execution.
FabricClient.invoke = invoke_safe_lro

result = run(spark, notebookutils, config)

# Print a concise JSON summary excluding the large definition payload
print(json.dumps({key: value for key, value in result.items() if key != "definition"}, indent=2))

# Safely display bindings only if present and non-empty
bindings = result.get("bindings")
if bindings:
    display(spark.createDataFrame(bindings))
else:
    print("No bindings to display.")



## Inspect or export the generated REST definition

The generated payload is retained in memory in `result['definition']`. The next cell only shows its part inventory. You can inspect each decoded TMDL part before applying.


In [ ]:
for part in result["definition"]["parts"]:
    print(part["path"])


## Materialize the graph (explicit opt-in)

After an APPLY run reports `saved_bindings_verified: true`, open `result['ontology_url']` and select **Manage graph → select the entity/relationship types → Continue → Materialize**. Confirm eligibility in that screen. This notebook does **not** create a separate graph item or invent an undocumented graph-materialization REST endpoint.

The default plan run is not proof of saved bindings or graph materialization. The APPLY run reads back `properties.generation` and the saved TMDL and fails if keys, source columns, schema names or relationship joins were lost. Actual graph materialization, source credentials, tenant rollout and runtime eligibility are still checked by Fabric.

If a run fails after creating junction tables, those verified Delta tables remain. Fix the reported cause and rerun. If ontology creation completed but the client lost the response, inspect the workspace instead of blindly resubmitting. A pre-existing ontology requires explicit `allow_update_existing = True`.

### References

- [Ontology definition — TMDL/new experience](https://learn.microsoft.com/rest/api/fabric/articles/item-management/definitions/ontology-definition)
- [Create ontology REST API](https://learn.microsoft.com/rest/api/fabric/ontology/items/create-ontology)
- [Update ontology definition](https://learn.microsoft.com/rest/api/fabric/ontology/items/update-ontology-definition)
- [Graph materialization and limitations](https://learn.microsoft.com/fabric/iq/ontology/how-to-use-ontology-graph)
